In [0]:
import os, sys, time
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "src"))
from lh import common, stream

# stream/01_run_stream (5.14): one short availableNow run of the payment-event stream (src/lh/stream.py).
#   layer = bronze (Kafka -> bronze.payment_events), silver (parse, de-duplicate, drop late), gold (failed payments per
#   customer per 10-minute window -> gold.payment_failure_windows), or all (the three in order; for dev by hand).
# The stream job runs bronze -> silver -> gold as three tasks. Kafka connection from the scope lh_<env> (never printed).
dbutils.widgets.dropdown("env", "dev", ["dev", "test", "prod"])
dbutils.widgets.dropdown("layer", "all", ["all", "bronze", "silver", "gold"])
env, layer = dbutils.widgets.get("env"), dbutils.widgets.get("layer")
cfg = common.load_config(env)
if cfg["code"] not in stream.load()["companies"]:
    raise ValueError(f"{cfg['code']} has no payment stream (config/stream.yml companies)")
scfg = stream.load()
secret = lambda k: dbutils.secrets.get(f"lh_{env}", k)
total = 0
for l in (["bronze", "silver", "gold"] if layer == "all" else [layer]):
    start = time.time()
    n = stream.run(spark, cfg, l, scfg, secret if l == "bronze" else None)
    total += n
    print(f"{l:7} {n:>7} rows added in {time.time() - start:.0f}s -> {cfg['catalog']}.{stream.TABLES[l]}")
    common.log_run(spark, cfg, f"stream.{l}", "success", n, f"availableNow run, {n} rows added")